# UrbanPulse Nepal — Municipality Profiles and Hierarchical Clustering

This notebook attempts to map each incident’s BIPAD ward identifiers to municipality identifiers using the official BIPAD ward and municipality API endpoints. If the lookup does not provide a verifiable municipality relationship, the notebook stops with the observed schema rather than joining on ambiguous place names.

When the mapping is available, incidents are aggregated to municipalities and represented by total count, hazard composition, and seasonal composition. Standardized profiles are compared with agglomerative clustering under several linkage and cluster-count choices. The analysis is descriptive of this retrieved snapshot; incomplete historical coverage and changes in reporting limit interpretation.

## 1. Load the processed incidents and configure API retrieval

Ward and municipality lookup records are saved separately from the incident snapshot. The API's own pagination links are followed, with retries, a timeout, and a page safety guard. The notebook records raw response pages and retrieval status for provenance.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse
import json, time
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').is_dir(): PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data').is_dir(): raise FileNotFoundError('Run from the repository or notebooks/ folder.')
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'bipad_incidents'
data_dirs = sorted((p for p in PROCESSED_ROOT.iterdir() if (p / 'incidents_processed.csv').is_file() and (p / 'data_quality_report.json').is_file()), key=lambda p:p.name)
if not data_dirs: raise FileNotFoundError('Run notebook 02 first; processed incident data is missing.')
DATA_DIR = data_dirs[-1]
incidents = pd.read_csv(DATA_DIR / 'incidents_processed.csv', low_memory=False)
API_BASE = 'https://bipadportal.gov.np/api/v1'
ENDPOINTS = {'ward':f'{API_BASE}/ward/', 'municipality':f'{API_BASE}/municipality/'}
MAX_PAGES_PER_ENDPOINT = 100
TIMEOUT_SECONDS = 45
retrieved_at = datetime.now(timezone.utc)
run_id = retrieved_at.strftime('%Y%m%dT%H%M%SZ')
RAW_LOOKUP_DIR = PROJECT_ROOT / 'data' / 'raw' / 'bipad_admin_lookup' / run_id
RAW_LOOKUP_DIR.mkdir(parents=True, exist_ok=False)
session = requests.Session()
retry = Retry(total=5, connect=5, read=5, status=5, backoff_factor=1, status_forcelist=(429,500,502,503,504), allowed_methods=frozenset({'GET'}), respect_retry_after_header=True)
session.mount('https://', HTTPAdapter(max_retries=retry))
session.headers.update({'Accept':'application/json','User-Agent':'UrbanPulseNepal-research/0.1'})
print('Incident snapshot:', DATA_DIR)
print('Ward endpoint:', ENDPOINTS['ward'])
print('Municipality endpoint:', ENDPOINTS['municipality'])

Incident snapshot: c:\Projects_Tracking\DisasterLens Nepal\data\processed\bipad_incidents\20261013T101245Z
Ward endpoint: https://bipadportal.gov.np/api/v1/ward/
Municipality endpoint: https://bipadportal.gov.np/api/v1/municipality/


## 2. Retrieve and inspect official administrative lookups

The helper accepts the common paginated `results` response or a plain list and follows same-host `next` links. A sentinel count is recorded but not treated as a trustworthy total. Empty results end retrieval; lookup completeness is recorded separately.

In [2]:
def fetch_endpoint(name, endpoint):
    records, pages, errors, seen = [], [], [], set()
    next_url, stop_reason, reported_total = endpoint, 'unknown', None
    for page_no in range(1, MAX_PAGES_PER_ENDPOINT+1):
        if not next_url:
            stop_reason = 'API pagination ended (no next link).'; break
        if next_url in seen:
            stop_reason = 'Repeated URL; stopped to prevent a pagination loop.'; errors.append({'page':page_no,'error':'Repeated URL'}); break
        if urlparse(next_url).netloc.lower() != urlparse(endpoint).netloc.lower():
            stop_reason = 'External pagination URL rejected.'; errors.append({'page':page_no,'error':'External URL'}); break
        seen.add(next_url)
        try:
            response = session.get(next_url, timeout=TIMEOUT_SECONDS); response.raise_for_status(); payload = response.json()
        except (requests.RequestException, ValueError) as exc:
            stop_reason = f'Request/JSON error on page {page_no}.'; errors.append({'page':page_no,'error':str(exc)}); break
        raw_path = RAW_LOOKUP_DIR / f'{name}_page_{page_no:04d}.json'
        raw_path.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')
        pages.append({'page':page_no,'url':next_url,'status_code':response.status_code,'file':raw_path.name})
        if isinstance(payload,dict) and isinstance(payload.get('results'),list):
            page_records = payload['results']; reported_total = reported_total if reported_total is not None else payload.get('count'); candidate_next = payload.get('next')
        elif isinstance(payload,list): page_records, candidate_next = payload, None
        else:
            stop_reason = f'Unexpected response structure on page {page_no}.'; errors.append({'page':page_no,'error':f'Unexpected payload type {type(payload).__name__}'}); break
        if not page_records:
            stop_reason = f'Empty results on page {page_no}.'; break
        records.extend(page_records)
        if not candidate_next:
            stop_reason = 'API pagination ended (no next link).'; break
        next_url = requests.compat.urljoin(response.url,candidate_next); time.sleep(0.15)
    else: stop_reason = f'MAX_PAGES_PER_ENDPOINT guard reached ({MAX_PAGES_PER_ENDPOINT}).'
    return records, {'endpoint':endpoint,'retrieved_at_utc':retrieved_at.isoformat(),'pages_saved':len(pages),'records_received':len(records),'api_reported_total':reported_total,'pagination_complete':stop_reason=='API pagination ended (no next link).' and not errors,'stop_reason':stop_reason,'errors':errors,'pages':pages}

ward_records, ward_manifest = fetch_endpoint('ward',ENDPOINTS['ward'])
municipality_records, municipality_manifest = fetch_endpoint('municipality',ENDPOINTS['municipality'])
(RAW_LOOKUP_DIR/'lookup_manifest.json').write_text(json.dumps({'ward':ward_manifest,'municipality':municipality_manifest},ensure_ascii=False,indent=2),encoding='utf-8')
ward_df, municipality_df = pd.DataFrame(ward_records), pd.DataFrame(municipality_records)
print('Ward retrieval:',{k:ward_manifest[k] for k in ['pages_saved','records_received','api_reported_total','pagination_complete','stop_reason']})
print('Municipality retrieval:',{k:municipality_manifest[k] for k in ['pages_saved','records_received','api_reported_total','pagination_complete','stop_reason']})
print('Ward fields:',sorted(ward_df.columns.astype(str)))
print('Municipality fields:',sorted(municipality_df.columns.astype(str)))
display(ward_df.head(3)); display(municipality_df.head(3))

Ward retrieval: {'pages_saved': 8, 'records_received': 6803, 'api_reported_total': 9223372036854775807, 'pagination_complete': False, 'stop_reason': 'Empty results on page 8.'}
Municipality retrieval: {'pages_saved': 2, 'records_received': 774, 'api_reported_total': 9223372036854775807, 'pagination_complete': False, 'stop_reason': 'Empty results on page 2.'}
Ward fields: ['bbox', 'centroid', 'id', 'municipality', 'title']
Municipality fields: ['bbox', 'centroid', 'code', 'district', 'id', 'order', 'title', 'title_en', 'title_ne', 'type']


,id,bbox,centroid,title,municipality
0,3302,"[88.03750747536112, 26.80199518868497, 88.1106...","{'type': 'Point', 'coordinates': [88.073051629...",3,3008
1,3309,"[88.08126638304078, 26.82418130845657, 88.1249...","{'type': 'Point', 'coordinates': [88.102203079...",6,3010
2,3746,"[85.38177269239043, 27.71640499112941, 85.3938...","{'type': 'Point', 'coordinates': [85.387038188...",7,27004


,id,bbox,centroid,title,title_en,title_ne,type,code,order,district
0,1005,"[87.6501351349468, 27.4059813235384, 88.201667...","{'type': 'Point', 'coordinates': [87.897130385...",Phaktanglung,Phaktanglung,फक्ताङलुङ,Rural Municipality,phaktanglungmun,1,1
1,1008,"[87.77471745301, 27.2923328214121, 88.15478069...","{'type': 'Point', 'coordinates': [87.946507034...",Sirijangha,Sirijangha,सिरीजङ्घा,Rural Municipality,sirijanghamun,2,1
2,3004,"[87.8412146034835, 26.8164192891186, 88.011316...","{'type': 'Point', 'coordinates': [87.908705743...",Ilam,Ilam,इलाम,Municipality,ilammun,3,3


## 3. Build an exact ward-to-municipality mapping

The join uses the incident `wards` identifiers and the administrative lookup IDs. It does not join on names. The code checks likely foreign-key field names and reports the returned fields if the ward endpoint does not expose a municipality relationship that can be interpreted safely.

In [3]:
def first_column(frame, candidates):
    return next((name for name in candidates if name in frame.columns),None)
def related_id(value):
    if isinstance(value,dict): return value.get('id',value.get('pk'))
    return value
def related_name(value):
    if isinstance(value,dict): return value.get('title',value.get('name',value.get('titleEn')))
    return None
ward_id_col = first_column(ward_df,['id','wardId','ward_id'])
municipality_fk_col = first_column(ward_df,['municipality','municipalityId','municipality_id','localLevel','localLevelId','local_level_id'])
municipality_id_col = first_column(municipality_df,['id','municipalityId','municipality_id'])
municipality_name_col = first_column(municipality_df,['title','name','titleEn','municipalityName'])
if not ward_id_col or not municipality_fk_col:
    raise ValueError(f'Cannot safely map ward IDs to municipalities. Observed ward fields: {sorted(ward_df.columns.astype(str))}. Inspect the saved lookup pages and update the explicit foreign-key mapping.')
if not municipality_id_col:
    raise ValueError(f'Cannot identify municipality ID in endpoint response. Observed municipality fields: {sorted(municipality_df.columns.astype(str))}.')
ward_map = pd.DataFrame({'ward_id':ward_df[ward_id_col].astype('string'), 'municipality_id':ward_df[municipality_fk_col].map(related_id).astype('string'), 'municipality_name_from_ward':ward_df[municipality_fk_col].map(related_name)})
municipality_map = pd.DataFrame({'municipality_id':municipality_df[municipality_id_col].astype('string'), 'municipality_name':municipality_df[municipality_name_col].astype('string') if municipality_name_col else municipality_df[municipality_id_col].astype('string')})
municipality_map = municipality_map.drop_duplicates('municipality_id')
ward_map = ward_map.dropna(subset=['ward_id','municipality_id']).drop_duplicates('ward_id')
ward_map = ward_map.merge(municipality_map,on='municipality_id',how='left')
ward_map['municipality_name'] = ward_map['municipality_name'].fillna(ward_map['municipality_name_from_ward'])
ward_map.to_csv(RAW_LOOKUP_DIR/'ward_to_municipality.csv',index=False,encoding='utf-8-sig')
print('Unique ward IDs mapped:',ward_map['ward_id'].nunique())
print('Unique municipalities mapped:',ward_map['municipality_id'].nunique())
print('Ward rows without municipality display name:',int(ward_map['municipality_name'].isna().sum()))
display(ward_map.head())

Unique ward IDs mapped: 6803
Unique municipalities mapped: 774
Ward rows without municipality display name: 0


,ward_id,municipality_id,municipality_name_from_ward,municipality_name
0,3302,3008,None,Rong
1,3309,3010,None,Suryodaya
2,3746,27004,None,Gokarneshwor
3,3310,3010,None,Suryodaya
4,866,64008,None,Sanni Tribeni


## 4. Aggregate incidents to municipality disaster profiles

An incident with more than one mapped ward is counted once per municipality, and unmapped incidents are reported. Features describe the distribution of recorded hazard IDs and seasons plus `log1p` of the municipality incident count. IDs stay categorical. The minimum incident thresholds are compared to show how sparse municipalities affect the analysis.

In [4]:
if 'wards' not in incidents.columns: raise ValueError('Processed incidents do not contain the BIPAD wards field.')
incident_rows = incidents[['id','wards','hazard_id','incident_season']].copy()
incident_rows['ward_id_list'] = incident_rows['wards'].map(lambda x: x if isinstance(x,list) else (json.loads(x) if isinstance(x,str) and x.startswith('[') else []))
incident_rows = incident_rows.explode('ward_id_list').rename(columns={'ward_id_list':'ward_id'})
incident_rows['ward_id'] = incident_rows['ward_id'].astype('string')
mapped = incident_rows.merge(ward_map[['ward_id','municipality_id','municipality_name']],on='ward_id',how='inner').drop_duplicates(['id','municipality_id'])
mapped_incident_ids = mapped['id'].astype('string').nunique()
all_incident_ids = incidents['id'].astype('string').nunique()
print(f'Incident IDs with at least one mapped municipality: {mapped_incident_ids:,} / {all_incident_ids:,} ({mapped_incident_ids/all_incident_ids:.1%})')
print('Distinct incident–municipality pairs:',len(mapped))
municipality_totals = mapped.groupby(['municipality_id','municipality_name'],dropna=False).size().rename('incident_count')
hazard_counts = pd.crosstab([mapped['municipality_id'],mapped['municipality_name']],mapped['hazard_id'].fillna('unknown'))
hazard_shares = hazard_counts.div(hazard_counts.sum(axis=1),axis=0).add_prefix('hazard_share_')
season_counts = pd.crosstab([mapped['municipality_id'],mapped['municipality_name']],mapped['incident_season'].fillna('Unknown'))
season_shares = season_counts.div(season_counts.sum(axis=1),axis=0).add_prefix('season_share_')
profiles = municipality_totals.to_frame().join(hazard_shares).join(season_shares).reset_index()
profiles['log1p_incident_count'] = np.log1p(profiles['incident_count'])
profiles = profiles.sort_values('incident_count',ascending=False).reset_index(drop=True)
display(profiles.head(10))
print('Municipality profiles before minimum-count thresholds:',len(profiles))

Incident IDs with at least one mapped municipality: 64,735 / 64,735 (100.0%)
Distinct incident–municipality pairs: 64735


,municipality_id,municipality_name,incident_count,hazard_share_2,hazard_share_3,hazard_share_4,hazard_share_5,hazard_share_6,hazard_share_7,hazard_share_8,...,hazard_share_21,hazard_share_22,hazard_share_23,hazard_share_24,hazard_share_45,season_share_Monsoon,season_share_Post-monsoon,season_share_Pre-monsoon,season_share_Winter,log1p_incident_count
0,5001,Belbari,1315,0.050951,0.0,0.000760,0.0,0.000000,0.008365,0.000760,...,0.0,0.002281,0.047148,0.022053,0.001521,0.257795,0.193916,0.304183,0.244106,7.182352
1,27005,Kageshwori Manahora,1307,0.006121,0.0,0.000000,0.0,0.000000,0.002295,0.003060,...,0.0,0.000765,0.002295,0.008416,0.003060,0.263963,0.196634,0.267789,0.271614,7.176255
2,4002,Barhadashi,1148,0.204704,0.0,0.000000,0.0,0.000871,0.002613,0.000871,...,0.0,0.000000,0.067073,0.055749,0.007840,0.325784,0.197735,0.261324,0.215157,7.046647
3,6009,Inaruwa,1035,0.046377,0.0,0.000000,0.0,0.000000,0.006763,0.002899,...,0.0,0.003865,0.028986,0.021256,0.005797,0.262802,0.178744,0.281159,0.277295,6.943122
4,15013,Rupani,826,0.036320,0.0,0.000000,0.0,0.002421,0.001211,0.001211,...,0.0,0.000000,0.033898,0.023002,0.008475,0.251816,0.145278,0.349879,0.253027,6.717805
5,33005,Jitpur Simara,797,0.031368,0.0,0.000000,0.0,0.002509,0.005019,0.001255,...,0.0,0.002509,0.035132,0.048934,0.000000,0.258469,0.168130,0.387704,0.185696,6.682109
6,27006,Kathmandu,793,0.005044,0.0,0.000000,0.0,0.000000,0.002522,0.003783,...,0.0,0.000000,0.002522,0.008827,0.003783,0.244641,0.214376,0.264817,0.276166,6.677083
7,40004,Pokhara,750,0.014667,0.0,0.005333,0.0,0.000000,0.014667,0.002667,...,0.0,0.000000,0.045333,0.032000,0.004000,0.302667,0.186667,0.308000,0.202667,6.621406
8,71005,Gauriganga,736,0.027174,0.0,0.002717,0.0,0.000000,0.001359,0.001359,...,0.0,0.001359,0.052989,0.089674,0.002717,0.364130,0.172554,0.349185,0.114130,6.602588
9,16007,Golbazar,735,0.027211,0.0,0.001361,0.0,0.009524,0.012245,0.001361,...,0.0,0.000000,0.020408,0.031293,0.005442,0.248980,0.168707,0.353741,0.228571,6.601230


Municipality profiles before minimum-count thresholds: 755


## 5. Compare agglomerative clustering configurations

Only municipalities meeting the stated minimum incident threshold are clustered. Numeric profile features are standardized. We compare Euclidean distance with Ward, average, and complete linkage across several cluster counts, and report silhouette scores and cluster-size balance. Silhouette is a diagnostic of these constructed profiles, not proof that the groups are naturally occurring categories.

In [5]:
feature_cols = [c for c in profiles.columns if c.startswith('hazard_share_') or c.startswith('season_share_')] + ['log1p_incident_count']
MIN_INCIDENTS_OPTIONS = [10,20]
N_CLUSTERS_OPTIONS = [3,4,5,6]
LINKAGES = ['ward','average','complete']
comparison_rows, label_outputs = [], []
for min_incidents in MIN_INCIDENTS_OPTIONS:
    subset = profiles.loc[profiles['incident_count'] >= min_incidents].copy().reset_index(drop=True)
    if len(subset) < max(N_CLUSTERS_OPTIONS)+1: continue
    X = StandardScaler().fit_transform(subset[feature_cols].fillna(0).to_numpy(dtype=float))
    for linkage in LINKAGES:
        for n_clusters in N_CLUSTERS_OPTIONS:
            if n_clusters >= len(subset): continue
            model = AgglomerativeClustering(n_clusters=n_clusters,linkage=linkage,metric='euclidean')
            labels = model.fit_predict(X)
            sizes = pd.Series(labels).value_counts()
            score = float(silhouette_score(X,labels,metric='euclidean'))
            comparison_rows.append({'minimum_incidents':min_incidents,'municipalities':len(subset),'linkage':linkage,'distance':'Euclidean on standardized features','n_clusters':n_clusters,'silhouette':score,'smallest_cluster_size':int(sizes.min()),'largest_cluster_size':int(sizes.max()),'cluster_size_ratio':float(sizes.max()/sizes.min())})
            label_outputs.append((min_incidents,linkage,n_clusters,subset[['municipality_id','municipality_name','incident_count']].assign(cluster=labels)))
comparison = pd.DataFrame(comparison_rows).sort_values('silhouette',ascending=False).reset_index(drop=True)
if comparison.empty: raise ValueError('Too few municipality profiles for the requested cluster comparisons.')
display(comparison)

,minimum_incidents,municipalities,linkage,distance,n_clusters,silhouette,smallest_cluster_size,largest_cluster_size,cluster_size_ratio
0,10,726,average,Euclidean on standardized features,3,0.771583,1,724,724.000000
1,10,726,complete,Euclidean on standardized features,3,0.771583,1,724,724.000000
2,20,610,average,Euclidean on standardized features,3,0.754706,1,608,608.000000
3,20,610,complete,Euclidean on standardized features,3,0.754706,1,608,608.000000
4,20,610,complete,Euclidean on standardized features,4,0.750854,1,607,607.000000
5,20,610,average,Euclidean on standardized features,4,0.750854,1,607,607.000000
6,20,610,average,Euclidean on standardized features,5,0.706409,1,606,606.000000
7,20,610,complete,Euclidean on standardized features,5,0.693895,1,606,606.000000
8,10,726,average,Euclidean on standardized features,4,0.691659,1,722,722.000000
9,20,610,average,Euclidean on standardized features,6,0.690239,1,605,605.000000


## 6. Profile candidate clusters and save results

Use the comparison table to identify reasonable candidates, then review their municipality counts and dominant profile features. The code selects no final model automatically; the highest silhouette can still correspond to imbalanced or hard-to-explain groups.

In [6]:
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'hierarchical' / DATA_DIR.name
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
profiles_path = OUTPUT_DIR / 'municipality_profiles.csv'
comparison_path = OUTPUT_DIR / 'hierarchical_configuration_comparison.csv'
profiles.to_csv(profiles_path,index=False,encoding='utf-8-sig')
comparison.to_csv(comparison_path,index=False,encoding='utf-8-sig')
label_frames=[]
for min_incidents,linkage,n_clusters,frame in label_outputs:
    tagged=frame.assign(minimum_incidents=min_incidents,linkage=linkage,n_clusters=n_clusters)
    label_frames.append(tagged)
labels_path=OUTPUT_DIR/'municipality_cluster_assignments.csv'
pd.concat(label_frames,ignore_index=True).to_csv(labels_path,index=False,encoding='utf-8-sig')
print('Municipality profiles:',profiles_path)
print('Configuration comparison:',comparison_path)
print('Candidate cluster assignments:',labels_path)
print('Lookup raw responses and manifest:',RAW_LOOKUP_DIR)

Municipality profiles: c:\Projects_Tracking\DisasterLens Nepal\data\processed\hierarchical\20261013T101245Z\municipality_profiles.csv
Configuration comparison: c:\Projects_Tracking\DisasterLens Nepal\data\processed\hierarchical\20261013T101245Z\hierarchical_configuration_comparison.csv
Candidate cluster assignments: c:\Projects_Tracking\DisasterLens Nepal\data\processed\hierarchical\20261013T101245Z\municipality_cluster_assignments.csv
Lookup raw responses and manifest: c:\Projects_Tracking\DisasterLens Nepal\data\raw\bipad_admin_lookup\20261013T103410Z


## Findings from the executed municipality analysis

### Administrative mapping and coverage

The BIPAD ward endpoint returned **6,803 ward records** and the municipality endpoint returned **774 municipality records**. All **64,735 incident IDs** in the saved incident snapshot mapped through a ward ID to an administrative municipality: **100% mapping coverage**, producing 64,735 distinct incident–municipality pairs. The incident data represented **755 municipalities**; 726 had at least 10 records and 610 had at least 20.

Both lookup endpoints returned the same implausible signed-integer maximum count (`9,223,372,036,854,775,807`) as the incident endpoint. Retrieval drained to an empty page (page 8 for wards; page 2 for municipalities), so the endpoint listings were saved, but the API total cannot verify completeness of administrative lookup coverage.

### Hierarchical comparison

The largest silhouette was **0.772** at minimum 10 incidents, average or complete linkage, and 3 clusters. Its cluster sizes were **724, 1, and 1** municipalities. With the 20-incident threshold, the highest score was **0.755**, again with a **608, 1, 1** split. These high scores reflect highly imbalanced partitions and are not useful evidence of coherent municipality groups.

Ward linkage was more balanced but had low silhouette scores (**0.059–0.098** across tested settings). For example, with minimum 20 incidents and 3 clusters, sizes were **314, 286, and 10**, with silhouette **0.097**. Increasing the cluster count introduced one-municipality groups in several configurations. Overall, the tested feature set did **not** establish a robust, interpretable final municipality clustering. The comparison is retained as an honest exploratory result rather than selecting a misleading “best” score.

### Interpretation limits

Profiles used incident totals, hazard-ID proportions, and seasonal proportions. Hazard IDs were not converted to labels, and each municipality’s observed records are not adjusted for population or reporting exposure. The incident archive’s historical completeness is also unverified. These groups, where produced, describe similarity in recorded profiles only; they are not measures of municipal risk or future disaster likelihood.